# NumJa GPU Fallback POC — JCuda/cuBLAS on NVIDIA T4

Future-only runbook for `bench.jcudapoc.GemmBench`: the production-HAL CPU baseline versus isolated JCuda 12.6.0/cuBLAS Dgemm.

Before running: in Colab select **Runtime > Change runtime type > T4 GPU**, then run all cells. This notebook is intentionally unexecuted evidence source; it contains no credentials and does not modify the TornadoVM POC notebook. The default benchmark size is N=4096.

## Step 0 — T4 preflight

Fail before building if Colab did not attach a T4. The later setup installs CUDA 12.6 because JCublas2 requires the CUDA 12 `libcublas.so.12` SONAME.

In [1]:
%%bash
set -eu
nvidia-smi | head -8
nvidia-smi --query-gpu=name,driver_version --format=csv,noheader | tee /tmp/jcuda-device-preflight.log
grep -qi 'T4' /tmp/jcuda-device-preflight.log || { echo 'ERROR: Colab T4 GPU is required.' >&2; exit 1; }


Sat Sep 12 16:21:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
Tesla T4, 580.82.07


## Step 1 — Future-only setup and Linux build

This cell installs JDK 21, Maven 3.9.15, and CUDA 12.6, writes the per-shell environment file, clones the Phase 7 branch over HTTPS, and builds on Linux so Maven resolves Linux-native JCuda artifacts. It does not download or manually install JCuda JARs.

In [2]:
%%bash
set -Eeu
trap 'echo "ERROR: setup failed at line $LINENO: $BASH_COMMAND" >&2' ERR

if ! java -version 2>&1 | grep -q '"21'; then
  apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jdk
fi
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

MVN_DIR=/opt/maven-3.9.15
if [ ! -d "$MVN_DIR" ]; then
  wget -q https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz -O /tmp/mvn.tgz
  tar -xzf /tmp/mvn.tgz -C /opt/
  mv /opt/apache-maven-3.9.15 "$MVN_DIR"
fi
export PATH="$MVN_DIR/bin:$PATH"

CUDA12_LIB=/usr/local/cuda-12.6/lib64
if [ ! -e "$CUDA12_LIB/libcublas.so.12" ]; then
  wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb
  dpkg -i /tmp/cuda-keyring.deb
  apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends cuda-toolkit-12-6
fi
test -e "$CUDA12_LIB/libcublas.so.12" || { echo "ERROR: missing $CUDA12_LIB/libcublas.so.12" >&2; exit 1; }

cat > /tmp/poc-env.sh <<EOF
export JAVA_HOME=$JAVA_HOME
export PATH=$MVN_DIR/bin:\$PATH
export LD_LIBRARY_PATH=/usr/local/cuda-12.6/lib64:\${LD_LIBRARY_PATH:-}
EOF
source /tmp/poc-env.sh

REPO=/content/java_ml
BRANCH=gsd/phase-07-gpu-fallback-poc-via-jcuda-cublas
if [ -d "$REPO/.git" ]; then
  git -C "$REPO" fetch origin "$BRANCH"
  git -C "$REPO" checkout "$BRANCH"
  git -C "$REPO" reset --hard "origin/$BRANCH"
else
  git clone --branch "$BRANCH" --single-branch https://github.com/minhhhduc/jml.git "$REPO"
fi
cd "$REPO"
mvn -q -pl modules/numja -am install -DskipTests
mvn -q -pl bench/jcuda-poc -am package -DskipTests
test -f bench/jcuda-poc/target/jcuda-poc-jar.jar
echo 4096 > /tmp/bench-n.txt
echo '=== SETUP OK: JDK 21, Maven 3.9.15, CUDA 12.6, Linux JCuda build ==='


Selecting previously unselected package cuda-keyring.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack /tmp/cuda-keyring.deb ...
Unpacking cuda-keyring (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
Selecting previously unselected package at-spi2-common.
(Reading database ... 126957 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package cuda-cccl-12-6.
Preparing to unpack .../01-cuda-cccl-12-6_12.6.77-1_amd64.deb ...
Unpacking cuda-cccl-12-6 (12.6.77-1) ...
Selecting previously unselected package cuda-cupti-12-6.
Preparing to unpack .../02-cuda-cupti-12-6_12.6.80-1_amd64.deb ...
Unpacking cuda-cupti-12-6 (12.6.80-1) ...
Selecting previously unselected package cuda-cupti-dev-12-6.
Preparing to unpack .../03-cuda-cupti-dev-12-6_12.6.80-1_amd64.deb ...
Unpacking cuda-cupti-dev-12-6 (12.6.80-1) ...
Selec

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/java_ml'...


## Step 2 — Benchmark, telemetry, and hard completion gate

`gpu_ms` is JCuda's kernel-only Dgemm timing (including device synchronization). `transfer_ms` is H2D plus D2H transfer time. Stderr is retained in `/tmp/poc-gpu.log` for native/device/number errors. The cell preserves the Java exit status and fails unless Java exits zero and the log contains exactly one `result=` line, `result=OK`; failures still preserve logs for D-09 NO-GO evidence.

In [ ]:
%%bash
set -eu
source /tmp/poc-env.sh
cd /content/java_ml
N=$(cat /tmp/bench-n.txt)
JAR=bench/jcuda-poc/target/jcuda-poc-jar.jar
test -e /usr/local/cuda-12.6/lib64/libcublas.so.12
test -f "$JAR"

rm -f /tmp/nv-sample.log /tmp/poc-gpu.log
nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used --format=csv,noheader,nounits -lms 100 -f /tmp/nv-sample.log &
NVPID=$!
trap 'kill "$NVPID" 2>/dev/null || true' EXIT
sleep 0.5
java -Dbench.env=colab -Dbench.size="$N" -jar "$JAR" 2>&1 | tee /tmp/poc-gpu.log
java_status=${PIPESTATUS[0]}
kill "$NVPID" 2>/dev/null || true
trap - EXIT
if [ "$java_status" -ne 0 ]; then
  echo "ERROR: GPU benchmark exited $java_status; preserve /tmp/poc-gpu.log as NO-GO evidence." >&2
  exit 1
fi
if [ "$(grep -c '^result=' /tmp/poc-gpu.log || true)" -ne 1 ] || ! grep -qx 'result=OK' /tmp/poc-gpu.log; then
  echo 'ERROR: GPU benchmark must emit exactly one result=OK; preserve /tmp/poc-gpu.log as NO-GO evidence.' >&2
  exit 1
fi

awk -F', ' '
{
  n++; util += $2; mem += $3
  if ($2 > utilPeak) utilPeak = $2
  if ($3 > memPeak) memPeak = $3
}
END {
  print "gpu_telemetry_samples=" n
  printf "gpu_util_peak_pct=%.0f", utilPeak; print ""
  printf "gpu_util_mean_pct=%.1f", n ? util / n : 0; print ""
  printf "gpu_mem_peak_mb=%.0f", memPeak; print ""
  printf "gpu_mem_mean_mb=%.1f", n ? mem / n : 0; print ""
  if (memPeak <= 0) exit 2
}' /tmp/nv-sample.log

## Step 3 — NumPy magnitude reference

This is a magnitude-only reference. Java `Random` and NumPy's generator produce different matrices, so `GemmBench`'s full-matrix Frobenius error is the authoritative accuracy check.

In [ ]:
import numpy as np
N = int(open('/tmp/bench-n.txt').read().strip())
a_np = np.random.default_rng(0xC0FFEE).random((N, N))
b_np = np.random.default_rng(0xBADF00D).random((N, N))
c_np = a_np @ b_np
print(f'N={N}')
print(f'||C_np||_F = {np.linalg.norm(c_np, "fro"):.6e}')
print(f'max |C_np| = {np.abs(c_np).max():.6e}')
print('Magnitude reference only; GemmBench Frobenius output is authoritative.')


## Step 4 — Decision summary and rubric

Copy the retained log, environment metadata, telemetry, and summary into `07-RESULTS.md`. Numerical gate: `cpu_vs_gpu_frob_rel_err <= 1e-9`. Performance GO requires both `speedup_ratio >= 2.0` and `transfer_pct < 50.0`; otherwise record NO-GO. Do not try a third GPU backend if this POC fails.

In [ ]:
%%bash
set -eu
echo '=== JCuda/cuBLAS timing samples (kernel-only GPU; transfer excluded) ==='
grep -E '^(cpu_sample_[1-3]_ms|gpu_sample_[1-3]_ms|cpu_baseline_ms|gpu_ms|transfer_ms|speedup_ratio|transfer_pct)=' /tmp/poc-gpu.log
echo '=== JCuda/cuBLAS result ==='
grep -E '^(env|device|jdk|hardware|size|cpu_vs_gpu_(frob_rel_err|max_abs_err|max_rel_err|mae)|result|verdict)=' /tmp/poc-gpu.log
echo '=== Telemetry ==='
awk -F', ' '
{
  n++; util += $2; mem += $3
  if ($2 > utilPeak) utilPeak = $2
  if ($3 > memPeak) memPeak = $3
}
END {
  print "gpu_telemetry_samples=" n
  printf "gpu_util_peak_pct=%.0f", utilPeak; print ""
  printf "gpu_util_mean_pct=%.1f", n ? util / n : 0; print ""
  printf "gpu_mem_peak_mb=%.0f", memPeak; print ""
  printf "gpu_mem_mean_mb=%.1f", n ? mem / n : 0; print ""
  if (memPeak <= 0) exit 2
}' /tmp/nv-sample.log
verdict=$(grep -oP '^verdict=\K.*' /tmp/poc-gpu.log | tail -1 || true)
frob=$(grep -oP '^cpu_vs_gpu_frob_rel_err=\K.*' /tmp/poc-gpu.log | tail -1 || true)
speedup=$(grep -oP '^speedup_ratio=\K.*' /tmp/poc-gpu.log | tail -1 || true)
transfer=$(grep -oP '^transfer_pct=\K.*' /tmp/poc-gpu.log | tail -1 || true)
printf 'Auto-verdict: %s\nFrobenius: %s\nSpeedup: %s\nTransfer: %s\n' "${verdict:-?}" "${frob:-?}" "${speedup:-?}" "${transfer:-?}"
case "$verdict" in GO) echo 'Recommendation: GO — consider later HAL integration.' ;; NO-GO) echo 'Recommendation: NO-GO — retain evidence and defer GPU.' ;; *) echo 'INSUFFICIENT_DATA — inspect /tmp/poc-gpu.log.' ;; esac